In [2]:
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd

In [60]:
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.106 Safari/537.36'}

page = "https://www.transfermarkt.us/as-saint-etienne/startseite/verein/618"
pageTree = requests.get(page, headers = headers)
pageSoup = BeautifulSoup(pageTree.content, 'html.parser')

In [61]:
PlayersList = []
AgeList = []
RolesList = []
PositionsList = []
NationList = []
ValuesList = []
PlayerLinks = []
TeamName = []
TeamNameFinal = []
NameID = []
PlayerID = []
TeamDetails = {}

Players = pageSoup.find_all("img", {"class": "bilderrahmen-fixed lazy lazy"})
Age = pageSoup.find_all("td", {"class": "zentriert"})
Roles = pageSoup.find_all("td", {"class": ["zentriert rueckennummer bg_Torwart", "zentriert rueckennummer bg_Abwehr", 
                                           "zentriert rueckennummer bg_Mittelfeld", "zentriert rueckennummer bg_Sturm"]})
Positions = pageSoup.find_all("tr", {"class": ["odd", "even"]})
Nationality = pageSoup.find_all("td", {"class": "zentriert"})
Values = pageSoup.find_all("td", {"class": "rechts hauptlink"})
Headers = pageSoup.find_all("li", {"class": "data-header__label"})
TeamName = pageSoup.find_all("h1", {"class": "data-header__headline-wrapper--oswald"})

In [62]:
for i in range(0, len(Players)):
    PlayersList.append(str(Players[i]).split('" class',1)[0].split('<img alt="',1) [1])
    
for i in range(1, (len (Players)*3),3):
    AgeList.append(str(Age[i]).split("(",1)[1].split(")",1)[0])
    
for i in range(0, len(Roles)):
    RolesList.append(str(Positions[i]).split('title="',1)[1].split('"><div')[0])
    
for i in range(2, (len (Players)*3),3) :
    NationList.append(str(Nationality[i]).split('title="',1)[1].split('"/',1)[0])
    
for i in range(0, len (Values)):
    if Values[i].text == '-':
        ValuesList.append('€0.0m')
    elif Values[i].text[-1] == 'k':
        millvalue = int(Values[i].text.strip('€').strip('k')) / 1000
        finalprice = f"€{str(millvalue)}m"
        ValuesList.append(finalprice)
    else:
        ValuesList.append(Values[i].text)
        
for i in range(0, len(Positions)):
    PositionsList.append(str(Positions[i].select_one('table.inline-table tr:nth-of-type(2) td').text.strip()))
    
for i in range(0, len(Positions)):
    pattern = r'href="(/[^"]+)"'
    playerlinks = re.findall(pattern, str(Positions[i]))
    if playerlinks[0].split('/')[1] != "profil" and len(playerlinks) >= 3:
        PlayerLinks.append(f"https://www.transfermarkt.us{playerlinks[2]}")
    else: 
        PlayerLinks.append(f"https://www.transfermarkt.us{playerlinks[0]}")
        
for i in range(0, len(Headers)):
    soup = BeautifulSoup(str(Headers[i]), 'html.parser')
    key = soup.find('li', class_='data-header__label').contents[0].strip(': \n')
    value = soup.find('span', class_='data-header__content').text.strip()
    TeamDetails[key] = value
    
Name = " ".join(TeamName[0].text.split())
                                      
TeamDetails.pop("Current transfer record")
string2 = TeamDetails['Stadium']
capacity = re.search(r'(\d{1,3}(?:\.\d{3})*) Seats', string2)
seats_number = capacity.group(1).replace('.', '')
TeamDetails['Capacity'] = seats_number
string = TeamDetails['Stadium'].split('\xa0\xa0')
TeamDetails['Stadium'] = string[0]
TeamDetails['Foreigners'] = TeamDetails['Foreigners'].split('\xa0\xa0')[0]

for i in range(0, int(TeamDetails['Squad size'])):
    TeamNameFinal.append(Name)

In [63]:
final_dt = pd.DataFrame({"Names":PlayersList,
                     "Age":AgeList,
                     "Team": TeamNameFinal,
                     "Position":PositionsList,
                     "Roles":RolesList,
                     "Nationality":NationList,
                     "Market Value (In Millions)":ValuesList,
                     "Player Profile Link":PlayerLinks})

for i in final_dt['Player Profile Link']: 
    NameID.append(i.split('/')[3])
    PlayerID.append(i.split('/')[6])

final_dt['Name ID'] = NameID
final_dt['ID'] = PlayerID

In [ ]:
imageURL = []

reader = pd.read_csv("Football-Data/Players_List.csv")
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.106 Safari/537.36'}
count = 0
for line in reader['Player Profile Link'][2621:2622]: 
    page = line
    pageTree = requests.get(page, headers = headers)
    pageSoup = BeautifulSoup(pageTree.content, 'html.parser')
    image_tag = pageSoup.find('img', class_='data-header__profile-image')
    if image_tag:
        image_url = image_tag['src']
    imageURL.append(image_url)
    count += 1

reader['Player Image'] = imageURL

In [ ]:
downloads_folder = os.path.expanduser("~/Downloads/intermediatepython-finalproj-kireetijosyula41/Football-Data")
file_path = os.path.join(downloads_folder, "Team_List.csv")
reader.to_csv(file_path, index=False)